# Red Cedar — Training Settings

This notebook shows how training settings change a model, one section per setting. Each section trains a small model (4 qubits, 1 layer) on the same dataset and tests it, so the effect of the setting is easy to see.

The steps:
1. `seed`: repeatable runs
2. `op_direction`: order of the operators
3. Settings are saved with the model
4. Settings in a file

The data is the Wisconsin breast cancer dataset (WDBC) from the server's catalog, the same data as the Quick Start.

Run the connect and setup cells first; after that, each section can be run on its own. All computation happens on the Red Cedar server; this notebook sends requests and shows the results. For the basic workflow see `01_quick_start.ipynb`; every setting is listed in the User Guide: [Settings](https://coherent-computing.github.io/redcedar/user_guide/#settings).

In [1]:
import redcedar

# Connect to a Red Cedar server: the one in REDCEDAR_URL (with the key in REDCEDAR_API_KEY)
# if those are set, otherwise the one on this machine, http://127.0.0.1:8002
client = redcedar.connect()
print(client.status())

# To choose a server in the code instead:
# client = redcedar.connect(url="https://your-server/api", api_key="your-key")

# Developers with the Red Cedar source installed (pip install -e) can use local mode,
# which runs Red Cedar in this process, with no server:
# client = redcedar.connect(local=True)

{'ok': True, 'server_version': '0.93-34-g806fa3e7', 'sdk_version': '0.94.0'}


In [2]:
# Setup: the dataset, the settings shared by every section, and a helper that trains and tests
dataset = "wdbc"

# Small, quick settings shared by every section; each section adds one setting.
base = {"num_qubits": 4, "min_layers": 1, "max_layers": 1}

def train_and_test(settings):
    """Train with these settings, test on the held-out data, return (train_id, accuracy, loss)."""
    result = client.training.run(dataset, settings=settings)
    test = client.training.evaluate(result["train_id"])
    return result["train_id"], test["overall_accuracy"], test["overall_loss"]

## 1. `seed`: repeatable runs

Training starts from random parameters, so two runs with the same settings usually end differently. `seed` fixes the starting parameters (and, on a CUDA-Q server, the circuit sampling), so a run can be repeated exactly. Leave it out, or set it to `None`, for a random start.

On a model this small the test accuracy often comes out the same; the loss shows the difference between runs.

In [3]:
runs = {
    "seed 123, run 1": dict(base, seed=123),
    "seed 123, run 2": dict(base, seed=123),
    "no seed":         dict(base),
}
results = {name: train_and_test(settings) for name, settings in runs.items()}
for name, (train_id, accuracy, loss) in results.items():
    print(f"{name:16s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

same = results["seed 123, run 1"][1:] == results["seed 123, run 2"][1:]
print(f"\nSame seed, same result: {same}")

seed 123, run 1  accuracy=75.4%  loss=0.268913  model 85af7d2c
seed 123, run 2  accuracy=75.4%  loss=0.268913  model f108dd7b
no seed          accuracy=75.4%  loss=0.252874  model a5cbe07f

Same seed, same result: True


## 2. `op_direction`: order of the operators

Each layer pairs the prediction qubit with every data qubit in turn. `op_direction` sets the order: `"backward"` (the default) starts at the highest data qubit, `"forward"` at the lowest. The number of parameters is the same either way; the results differ because the optimizer visits the parameters in a different order.

Both runs below use the same seed, so the direction is the only difference. Neither direction is better in general: across many seeds and datasets the results were close.

In [4]:
for direction in ["backward", "forward"]:
    train_id, accuracy, loss = train_and_test(dict(base, seed=123, op_direction=direction))
    print(f"{direction:8s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

backward accuracy=75.4%  loss=0.268913  model a91771c8


forward  accuracy=75.4%  loss=0.267467  model 0c09f6e9


## 3. Settings are saved with the model

Each trained model keeps its settings, and the model list shows them, so any run can be traced back and repeated.

In [5]:
# The quantum models trained on this dataset, newest first. dataset_name is the dataset's file
# on the server (wdbc.csv for the catalog's WDBC)
models = [m for m in client.inference.models()["models"]
          if dataset in str(m["dataset_name"]).lower() and m.get("model", "quantum") == "quantum"]
for m in models[:5]:
    p = m["training_params"]
    print(f"{m['train_id'][:8]}  {m['dataset']:12s} seed={p['seed']}  op_direction={p['op_direction']}")

0c09f6e9  WDBC (Breast Cancer) seed=123  op_direction=forward
426f0633  WDBC (Breast Cancer) seed=123  op_direction=forward
84b4e3db  WDBC (Breast Cancer) seed=123  op_direction=backward
85af7d2c  WDBC (Breast Cancer) seed=123  op_direction=backward
9a468750  WDBC (Breast Cancer) seed=123  op_direction=backward


## 4. Settings in a file

Settings can also be kept in a JSON or TOML file and passed by its path. A settings file holds settings only, not the data, so the same file works for any dataset. The `settings/` folder has two: `quantum.json` with the small settings used above, and `forest.json`, which trains a random forest (`"model": "random_forest"`, its own settings in `forest_options`; see `06_random_forest.ipynb`).

The same files work from the command line: `redcedar train data/wdbc_train.csv --settings settings/forest.json`. More in the User Guide: [Passing Settings](https://coherent-computing.github.io/redcedar/user_guide/#passing-settings).

In [6]:
import json

for path in ["settings/quantum.json", "settings/forest.json"]:
    with open(path) as f:
        print(f"{path}: {json.load(f)}")
    train_id, accuracy, loss = train_and_test(path)   # settings= takes the file's path
    print(f"  accuracy={accuracy:.1%}  model {train_id[:8]}")

settings/quantum.json: {'num_qubits': 4, 'min_layers': 1, 'max_layers': 1, 'seed': 123}


  accuracy=75.4%  model 774f4da5
settings/forest.json: {'model': 'random_forest', 'seed': 123, 'forest_options': {'n_estimators': 300, 'min_samples_leaf': 5}}


  accuracy=94.7%  model 10846a49
